# 02.2 Numerical Dtypes & Promotion under NEP 50

> **Prerequisites:** 02.1 (dtype as the buffer's interpretation; views and the migration
> discipline) · 01.2 (units and the ingestion contract — dtypes are the sibling failure)
> **What you'll learn:**
> - Price a dtype in domain terms — what int32 and float32 can and cannot hold of PayFlow's money — before a column ships
> - Predict 2.x promotion outcomes from the weak-scalar/strong-scalar rule instead of 1.x value-based folklore
> - Name the three behaviour changes a 1.x→2.x port must audit for, and reproduce each on demand
> - Distinguish a dtype that is wrong from an algorithm that is covering for it, using the float32 money sum
> - Write the dtype contract that fails a doomed column at ingestion instead of at month-end
> **Level:** Beginner · **Series:** 02 NumPy & Vectorized Computing

> ⚡ **Wednesday 2026-07-15, 06:30** — the first nightly run after the platform upgrade, and
> the INR billings panel jumps from roughly twenty-one million rupees to fifteen and a half
> billion. Finance freezes the dashboard and the upgrade is presumed guilty — same data, same
> query, new number. The cause: the upgrade did not break the total. It fixed one that had
> been wrapping, silently, for years — on this operating system and no other.


## Concept
### Plain-English Explanation

02.1 established that a dtype tells numpy how to *read* bytes. This notebook is about the
other half of the job: a dtype also decides how arithmetic on those bytes *behaves* — how high
a number can climb before it wraps around to garbage, how finely a value can be represented
before pennies stop existing, and what type comes out when two different types meet. None of
these decisions raise errors when they go wrong. An integer that exceeds its range wraps; a
float that exceeds its precision rounds; a mixed-type expression silently picks a winner. All
three produce plausible numbers, which by now the reader will recognise as this track's
definition of dangerous.

The cold open is the first kind. PayFlow's paise column was stored as a 32-bit integer —
every individual invoice fit comfortably — but the *sum* of the column lapped the 32-bit
ceiling 361 times, and on Windows under numpy 1.x the sum's accumulator was 32-bit too. The
dashboard had shown the leftover of a modular division for years. What changed on upgrade
night was not the data; it was numpy 2.x making the default integer 64-bit on Windows, which
quietly produced the first correct total the panel had ever displayed.

### Technical Explanation

Three groups of facts organise everything below. **Ranges and spacing.** An integer dtype
holds a hard interval — int32 tops out at 2,147,483,647, which in paise is about INR 21.5
million, a number one enterprise account can exceed in a year — and arithmetic past the edge
wraps modulo 2ⁿ. A float dtype holds a huge range thinly: float32 near 1e8 has a gap of 8
between adjacent representable values, so it cannot count paise at PayFlow's scale even
though 1e8 is nowhere near its maximum. float64's gap at the same magnitude sits near a hundred-millionth of a rupee. ⭐
**CRITICAL CONCEPT** — a float's failure mode is not overflow but *spacing*: it does not run
out of room, it runs out of resolution, and it does so silently and gradually.

**Promotion.** When dtypes meet, numpy must choose the result type. The 2019 notes were
written against *value-based* casting: the value of a scalar could demote or promote the
result, so `uint8_array + 200` widened to uint16 because 400 needs it. **NEP 50**, the 2.x
rule, removed value-sensitivity: a Python scalar is **weak** (it adopts the array's dtype),
a numpy scalar is **strong** (it promotes like an array of its type). Results now depend
only on types — predictable, but different from 1.x in three specific, auditable ways the
build reproduces.

**Platform history.** The **default integer** — what `np.array([1])` gives you, and what
integer reductions accumulate in when the input is narrower — was the C `long` in 1.x:
64-bit on Linux, **32-bit on Windows**. numpy 2.x made it int64 everywhere. That single line
of release notes is the cold open, and its trap survives the fix: `np.long` still exists and
still means the C long, which on this machine is int32 wearing a 64-bit-sounding name.

### Mental Model

A dtype is a budget with two lines — magnitude and resolution — and promotion is the rule
for merging budgets. Every numeric column should be able to answer: what is the largest
value, and the largest *sum*, this dtype will ever be asked to hold, and what is the
smallest difference it must preserve? If nobody has answered, the column is a wrap or a
rounding on a schedule.


## How It Works

```text
  int32 as a circle (two's complement): the sum walks clockwise and LAPS

        0 ────────▶ +2,147,483,647
        ▲                 │ +1 wraps to
        │                 ▼
        −1 ◀──────── −2,147,483,648

  paise column: 116,888 invoices, total 1,552,569,768,898
  int32 accumulator: 1,552,569,768,898 mod 2^32 laps ... 361 full laps
                     leftover shown on the dashboard: 2,086,575,042

  NEP 50 promotion (numpy 2.x):
    array_dtype (X) + python scalar  ──▶  X          (scalar is WEAK: adopts X)
    array_dtype (X) + numpy scalar Y ──▶  promote(X, Y)   (Y is STRONG)
    python int too big for X         ──▶  OverflowError, loudly

  1.x value-based (what the 2019 notes assume):
    result type could depend on the VALUE of the scalar - uint8+200 widened to uint16;
    np.float64(3) got DEMOTED next to a float32 array. Both are gone.
```

The circle is the entire incident. Two's-complement addition is modular arithmetic: the
accumulator is a register that walks a circle of 2³² positions, and the trillion-and-a-half-paise
walk goes around 361 times before stopping at 2,086,575,042 — the number finance saw. Nothing
about the walk raises: every intermediate step is a legal int32. The correctness of an integer
sum is therefore a property of the *accumulator's* dtype, not just the elements' — the
elements all fit, and the total still could not.

Where the accumulator comes from is the platform history above. Integer reductions accumulate
in at least the default integer; under 1.x-on-Windows that meant int32, and the same script
on a colleague's Linux box gave the right answer, which is part of why the bug survived —
every attempt to reproduce it elsewhere failed. numpy 2.x's int64-everywhere default is what
changed the dashboard, and the build below reproduces both worlds explicitly on the pinned
install.

The promotion block explains the other half of the port's audit surface. Weak Python scalars
mean `float32_array + 3.0` stays float32 — unchanged from 1.x, and cheap. Strong numpy
scalars mean `float32_array + np.float64(3)` is now float64 — *more* precision than 1.x gave,
a silent improvement. The dangerous quadrant is unsigned arithmetic: `uint8_array + 200`
stays uint8 and wraps silently where 1.x widened, because 200 fits uint8 and the result's
value no longer gets a vote. And a Python scalar that does not fit the array's dtype at all
now refuses loudly instead of widening. One improvement, one refusal, one new silent wrap —
a port that audits only for exceptions finds the refusal and misses the wrap.


## Hands-On Build
### Stage A — from scratch

Collapsed (library-API notebook): dtypes are numpy's own vocabulary — the raw mechanism here
is two's-complement and IEEE-754 arithmetic, which 02.1's buffer work already grounded and
whose full derivation belongs to the numerical-precision treatment in series 05.

### Stage B — idiomatic

First the bestiary, priced in PayFlow's own units rather than abstract powers of two —
because "int32 holds roughly two billion" lands differently when written as "int32 holds
INR 21.5 million of paise".


In [1]:
# Load the committed lab module; it owns the stdlib parse of the INR invoice lane
# (M7 commas stripped at the boundary, M6 casing preserved on purpose for the
# StringDType block) and every experiment this notebook shows.
import importlib.util
import sys
from pathlib import Path

import numpy as np

LAB = Path.cwd() / "_lab" / "lab_02.2_dtypes_nep50.py"
spec = importlib.util.spec_from_file_location("lab_02_2", LAB)
lab = importlib.util.module_from_spec(spec)
sys.modules["lab_02_2"] = lab
spec.loader.exec_module(lab)

d = lab.load_inr_invoices()
print(f"INR lane: {len(d['amount_inr']):,} invoices   amount dtype "
      f"{d['amount_inr'].dtype}   dates {d['issue_date'].dtype}\n")
lab.dtype_bestiary()

INR lane: 116,888 invoices   amount dtype float64   dates datetime64[D]

  dtype      bytes                 max value   holds, in paise
  int8           1                       127   INR 1
  int16          2                    32,767   INR 328
  int32          4             2,147,483,647   INR 21,474,836
  int64          8 9,223,372,036,854,775,807   INR 92,233,720,368,547,760
  float16        2                 ~6.55e+04   cannot represent 1e8 at all (max ~6.55e+04)
  float32        4                  ~3.4e+38   gap at 1e8: 8
  float64        8                 ~1.8e+308   gap at 1e8: 1.49012e-08

  int32's ceiling is INR 21.5 million in paise - ONE large enterprise invoice
  cleared with a year of billing behind it. float32 near 1e8 cannot even
  represent every paisa: adjacent values are 8 apart. Money wants int64 minor
  units or float64 - everything narrower is a wrap or a rounding waiting to happen.


Read the table as a set of ceilings PayFlow will actually hit. int16 cannot hold one
mid-sized invoice in paise. int32 holds any single invoice and cannot hold a week of them —
its ceiling, INR 21.5 million, is the incident's entire setup. float16 cannot represent 1e8
*at all*. And the float32 row is the subtle one: its maximum is astronomically far away, but
its **gap at 1e8 is 8** — near a hundred million, values exist only every 8 units, so paise
at billing scale simply stop being distinct. Ranges fail loudly at the edge; spacing fails
quietly everywhere past the resolution.

Next, promotion — the rules that decide what happens when the dtypes above meet a scalar or
each other, and the exact places those rules changed under the 2019→2026 port.


In [2]:
lab.promotion_table()

  expression                      numpy 2.x (captured)              1.x value-based (documented)
  np.uint8(200) + 200             uint8 144                         uint16 400
  array([200], uint8) + 200       uint8 144 (SILENT)                uint16 400
  array float32 + 3.0 (py float)  float32 4.5                       float32 4.5 (same)
  array float32 + np.float64(3)   float64 4.5                       float32 4.5
  array([100], int8) + 300        OverflowError - refuses           int16 400

  the rule that replaced value-based casting: python scalars are WEAK (they
  adopt the array's dtype); numpy scalars are STRONG (they promote like any
  array). Results no longer depend on the VALUE of a scalar - only on types.
  The three behaviour changes above are exactly the port's audit surface:
  in-range uint arithmetic now wraps silently where 1.x widened; out-of-range
  python ints now raise; and a float64 numpy scalar now wins, ADDING precision.


The middle column is executed on the pinned numpy 2.5.2; the right column is what the
same expressions produced under 1.x value-based casting — documented behaviour from the NEP
50 migration guide, labelled as such rather than pretended captured (there is no 1.x in this
environment to run). The audit surface for any port is exactly the rows where the columns
differ. **Row two is the dangerous one**: in-range unsigned arithmetic now wraps *silently*
in arrays — 200 + 200 = 144 with no warning — where 1.x widened to uint16. Row five is the
loud one: a Python int that cannot fit the array's dtype refuses outright, so 1.x code that
leaned on value-based widening now crashes, which is at least visible. Row four is the quiet
improvement: a strong `np.float64` scalar now wins the promotion, where 1.x demoted it to
float32 — same code, *more* precision after the port.

⚠️ The transferable rule beats memorising rows: **Python scalars are weak, numpy scalars are
strong, and values no longer vote.** Every cell in the table is a one-line application of
that sentence.

Ranges wrap; precision *rounds*. The float32 block below is the same money on the other
failure axis — and a warning about being saved by an algorithm without knowing it.


In [3]:
lab.float32_money(d["amount_inr"])

  float64 truth, full INR lane      :    15,525,697,688.98
  float32, numpy pairwise reduction :    15,525,697,536.00   off by +152.98
  float32, naive running loop (50k) :     6,586,237,952.00   off by -28,160.00 on the same 50k rows
  (float64 truth on those 50k rows  :     6,586,209,725.46)

  numpy's pairwise summation hides most of float32's sins - until someone
  'optimizes' into a running accumulator, a streaming job, or a GPU kernel
  that sums naively. The dtype was never safe; the algorithm was covering.


Three sums of the same column. In float64 the lane totals 15,525,697,688.98. Cast to
float32 and let numpy sum it, and the answer is off by only about 153 rupees in fifteen and a
half billion — seemingly fine. But numpy's reduction is **pairwise**: it sums in a tree, so
rounding errors grow logarithmically rather than linearly, and that algorithm — not the dtype
— is doing the protecting. The third line removes the protection: a plain running accumulator
in float32, the shape every streaming job and hand-rolled loop uses, drifts by about
twenty-eight thousand rupees on just the first fifty thousand rows. Once the running total
reaches the billions, float32's spacing at that magnitude is hundreds of rupees wide, so
every further addition rounds by up to half that gap — and fifty thousand roundings, no
longer cancelling cleanly, become the drift. ⚠️ The trap is that both float32 pipelines *validate* against each other in a small
test and diverge at scale — the dtype was never adequate; the algorithm was covering, and only
while the code path used it. Money belongs in int64 minor units or float64, as policy rather
than as tuning.

Two more dtypes round out the toolkit before Stage C: dates and text, both of which the 2019
notes handled as generic objects and 2.x handles as first-class dtypes.


In [4]:
lab.datetime_lane(d["issue_date"], d["due_date"])
print()
lab.string_lane(d["status"])

  issue_date dtype datetime64[D]: an int64 count of DAYS since the epoch,
    wearing calendar clothes - arithmetic is integer arithmetic, no float error
  payment terms = due - issue: 15d x 64,322, 30d x 38,875, 45d x 13,691
  NaT is datetime64's NaN: NaT == NaT -> False, comparisons all False; use np.isnat() - same discipline as np.nan (02.5)
  unit matters: datetime64[D] cannot hold 09:40; [ns] spans only 1678-2262.
  choose the coarsest unit the domain needs - invoices are day-grained.

  numpy's inferred dtype for the status column: <U11
  fixed-width <U11:   5.1 MB  (11 UCS-4 slots x 116,888 rows, padding included)
  StringDType    : array header   1.9 MB + heap strings (variable, no padding)
  M6 in one line: status == 'paid' matches 101,102 rows raw,
  112,084 after np.strings.lower - 10,982 paid invoices hidden by casing
  np.strings is the 2.x home for vectorized text ops (np.char and chararray
  are the deprecated 2019-era spellings); heavy text work belongs to pandas (03).


`datetime64[D]` is an int64 count of days wearing calendar clothes — subtraction is
exact integer arithmetic, and the payment-terms histogram (15, 30 and 45 days, in that order
of volume) falls out of one vectorized subtraction. Two disciplines carry over wholesale:
**NaT** is datetime's NaN, unequal even to itself, probed with `np.isnat` (02.5 develops
this); and the **unit** is part of the type — `[D]` cannot hold a timestamp's 09:40, while
`[ns]` spans only the years 1678–2262. Choose the coarsest unit the domain needs; invoices
are day-grained.

The string block is the 2.x story in miniature. numpy's inferred `<U11` pads every status to
eleven UCS-4 slots — 5.1 MB for a column whose content is a few short words — while 2.x's
**StringDType** stores variable-length strings on a heap behind a 1.9 MB array of references.
`np.strings.lower` then does in one vectorized call what M6 has been threatening since 01.2:
`status == "paid"` matches 101,102 rows raw and 112,084 after lowercasing — 10,982 paid
invoices hidden by casing, the same defect 01.2's contract counted, now fixed idiomatically.
(`np.char` and `chararray` are the 2019-era spellings; the latter is formally deprecated in
numpy 2.5. Heavy text work belongs to pandas, in series 03.)

### Stage C — production

Every failure above was a column that could not hold its future. The production artifact is
the dtype contract: a per-column statement of type, headroom and resolution, enforced where
01.2's grain and unit checks run — at ingestion, before habits form on top of a doomed column.


In [5]:
lab.contract_demo(d["amount_inr"], d["issue_date"])

  policy: money is int64 minor units or float64 - never narrower, never unsigned

  [PASS] amount_paise as int64
  [FAIL] amount_paise as int32 (the incident's column)  FAIL: dtype int32 != contracted int64; TOTAL exceeds 0.1% of dtype range - accumulator wrap risk
  [PASS] amount_inr as float64
  [FAIL] amount_inr as float32                          FAIL: dtype float32 != contracted float64; representable gap 0.25 at max - cannot hold minor units
  [PASS] issue_date as datetime64[D]

  the contract runs where 01.2's grain and unit checks run - at ingestion,
  before any consumer can build a habit on a column that cannot hold its future


The contract asks two questions the incident's column would have failed years early:
is the dtype the contracted one, and — the line schemas never carry — does the dtype hold
the column's **future**, not just its elements? The headroom rule prices the *total*: an
integer money column must fit its own sum with three orders of magnitude to spare, and the
int32 paise column fails it immediately ("accumulator wrap risk"), while the float rule
prices resolution — float32's representable gap of 0.25 at the column's maximum cannot hold
minor units. Note what makes this cheap: every check is one `np.iinfo`/`np.spacing` call
against data already in memory at ingestion.

## Evaluation

As in 02.1, the harness is assertion-shaped (guide §2: toolkit notebooks may evaluate with
unit-test-style checks). Three layers are captured above: the **promotion table**, which is
an executable regression — rerun it on any numpy upgrade and a changed cell is a changed
promotion rule, exactly how the 2.x differences would have been caught before production
found them; the **incident pair**, where the same column sums two ways and the int64/int32
disagreement *is* the detection; and the **contract verdicts**, five PASS/FAIL lines whose
flipping is the alert. Nothing here is stochastic, so there is no noise band — a delta on
any line is signal by construction.


## Design Patterns / Tradeoffs

**Integer minor units versus float64 major units for money.** int64 paise are exact: sums,
differences and equality behave like ledger arithmetic, audits reconcile to zero, and the
headroom is nine trillion rupees of paise — but every interest rate, FX conversion or
percentage requires an explicit rounding policy at each step, and mixed-currency work needs
the unit carried alongside (01.2's `Money` pair, one layer down). float64 rupees absorb
division and rates gracefully and hold PayFlow's magnitudes with spacing far below a paisa — but
equality becomes tolerance-based and cent-exact reconciliation needs care. Use int64 minor
units where money is *stored* or *reconciled*; float64 where it is *modelled*; never float32
for either, and never unsigned — a refund made PayFlow's smallest payment negative the week
someone tried.

**Narrow dtypes as compression versus as arithmetic.** Casting a feature matrix to float32
or int16 halves memory and can double throughput (02.7 measures this), and for *storage and
model inputs* it is often right — ML features tolerate 1e-7 relative noise. The failure is
letting the narrow dtype do *aggregation*: the incident column was fine as storage and wrong
the moment it met its own sum. Narrow at rest, widen to accumulate — and write the widening
into the code (`sum(dtype=np.int64)`) rather than trusting defaults that just proved
platform-dependent.

**Auditing a 1.x→2.x port: grep versus regression table.** Grepping for removed names
(`np.int`, `np.unicode_`, `.dtype =` assignments) catches the loud breaks — they raise. The
behaviour changes that matter are the silent ones, and they are enumerable: unsigned
arithmetic with in-range Python scalars, numpy-scalar promotions, and integer defaults on
Windows. A ten-line executed promotion table per pinned environment turns "we think the port
is safe" into a diffable artifact — the same move 01.3 made for reproducibility.

**Recommendation for PayFlow:** money columns are int64 paise at rest with explicit int64
accumulators, float64 in models; the dtype contract runs at ingestion beside 01.2's checks;
the promotion table lives in the repo and runs in CI against the pinned numpy; `np.long` and
friends are banned by lint in favour of explicit-width names.


## Production Scenario
### Symptoms

**Wednesday 2026-07-15, 06:30.** The platform upgrade that pinned numpy 2.x landed overnight.
The nightly billings job runs on schedule.

- **06:30** — the INR billings panel reads fifteen and a half billion rupees against
  yesterday's twenty-one million — a number so large the panel's y-axis re-scales. Every
  other currency lane is unchanged.
- Finance freezes the dashboard and files against the upgrade: same exports, same SQL, same
  job code — the only diff in a fortnight is the dependency pin.
- The job's logs are clean before and after the upgrade. No overflow warning appears in
  either era, and row counts match to the invoice.
- A developer reruns yesterday's *pre-upgrade* environment on today's data: twenty-one
  million again. The number tracks the numpy version, not the data — reproducibly.
- The team's Linux staging box has *always* shown the fifteen-billion figure, a discrepancy
  noted in a two-year-old ticket titled "staging INR total wrong?" and closed as
  environment noise.


In [6]:
summary = lab.incident(d["amount_inr"])

  rows in the INR lane: 116,888
  paise total, numpy 2.x default (int64 accumulator):      1,552,569,768,898
  the stored column is int32 (every element fits); numpy 2.x's DEFAULT sum
  of it accumulates in int64 and agrees:           1,552,569,768,898
  the same sum under an int32 accumulator (1.x-on-Windows behaviour,
  reproduced explicitly here):                                 2,086,575,042
  the 'revenue jump' the upgrade produced:                 1,550,483,193,856
  = the accumulator lapped 2^32 361 times; the OLD number was the
    broken one, and it had been broken silently for years

  the attempted hotfix: 'cast it to np.long, that is 64-bit'
  np.long on this Windows box is int32 (32-bit) - the C long, NOT a guaranteed 64 bits
  sum(dtype=np.long) on the int32 column:                      2,086,575,042
  identical wrap. The unambiguous spelling is np.int64; np.long is a
  platform question wearing a dtype's name.


### Diagnosis

Walking the ladder in its numerical-incident form, naming what each signal eliminated:

1. **Alert** — a 744-fold jump in one panel after a dependency upgrade. Candidate causes: the
   upgrade broke the computation, the upgrade *fixed* the computation, or the data moved.
2. **Input checks** — exports hash identical to the pre-upgrade run's manifest (01.3), and
   the non-INR lanes agree across environments. Data eliminated; the jump is confined to the
   one lane whose magnitude is three orders larger in local units (01.2's M2 geometry).
3. **Reproduce both eras** — the cell above: the stored int32 column summed with an int64
   accumulator gives 1,552,569,768,898; summed with an int32 accumulator it gives
   2,086,575,042 — yesterday's dashboard number, exactly. This converts "which release broke
   it" into "which of two arithmetics is *right*", a question with an objective answer.
4. **Arithmetic check** — the total exceeds int32's ceiling by a factor of ~723, so a 32-bit
   accumulator must wrap; it lapped 2³² precisely 361 times. The old number is the leftover
   of a modular division. The *new* number is the first correct one. The Linux "discrepancy"
   ticket was the truth, filed as a bug.
5. **Mechanism named** — numpy 1.x integer reductions accumulate in the platform's C long:
   32-bit on Windows, 64-bit on the Linux staging box — which is why the bug lived only on
   the Windows production host and every cross-check elsewhere "failed to reproduce". numpy
   2.x made the default integer int64 on Windows too; the upgrade repaired the sum as a side
   effect.
6. **The hotfix trap, pre-empted** — the reflex patch `sum(dtype=np.long)` is measured in the
   same cell: `np.long` is the C long, int32 on this box, and reproduces the wrap to the
   paisa. Width by *name* (`np.int64`), never by platform alias.

### Root Cause

The paise column was stored as int32 — every element fit — and under numpy 1.x on Windows
integer sums accumulated in the 32-bit C long, so the column's 1,552,569,768,898-paise total wrapped
modulo 2³² and the dashboard displayed the 2,086,575,042 remainder for years. The upgrade to
numpy 2.x changed the Windows default integer to int64, silently correcting the total and
exposing the historical wrap as an apparent regression.

### Fix

**Mitigation now.** Unfreeze the dashboard with the new number annotated as corrected, and
restate the historical series by re-summing archived exports with an explicit int64
accumulator — the archives are intact; only the arithmetic was wrong.

**Permanent fix.** Money columns move to int64 minor units at rest (the Stage C contract's
first line), and every integer reduction on money states its accumulator:
`paise.sum(dtype=np.int64)`. The dtype contract joins ingestion so the next
column-that-cannot-hold-its-future fails on arrival, and the promotion table joins CI so the
next numpy upgrade diffs its arithmetic before production does.

### Prevention

- **Headroom is part of schema review**: a numeric column ships with its maximum plausible
  *sum*, not just its maximum element — the contract's 1000× rule automates the question.
- **Accumulators are explicit on money paths.** Defaults just proved platform-dependent;
  `dtype=np.int64` is four characters of insurance per reduction.
- **Cross-platform disagreement is a finding, never noise.** The two-year-old staging ticket
  contained the entire incident; a policy of reconciling environment discrepancies to a
  mechanism would have caught this in 2024.
- **Pin-upgrade CI runs the promotion table** and the era-pair sum, so arithmetic changes
  arrive as a red diff instead of a frozen dashboard.


## Common Pitfalls

⚠️ **Trusting the elements to certify the sum.** Every paise value fit int32; the total
lapped it 361 times. Overflow on aggregation is a property of the accumulator, so ask the
headroom question of the *sum*, and state the accumulator dtype on money paths.

⚠️ **`np.long` as "long = 64-bit".** It is the C long: int32 on Windows, int64 on Linux —
a platform question wearing a dtype's name, and this machine's `np.long` reproduces the
incident's wrap exactly. Use explicit widths (`np.int64`); leave the C-family names to FFI.

**Porting 1.x unsigned arithmetic unaudited.** `uint8_array + 200` widened under value-based
casting and now wraps silently to 144. The loud breaks of the 2.x port announce themselves;
this one only changes numbers. Audit unsigned code paths by hand or by regression table.

**float32 for money because "the tests passed".** numpy's pairwise sum hid the damage; the streaming rewrite's running accumulator did not, drifting tens of thousands of rupees
across fifty thousand rows. When a narrow dtype looks fine, first ask which *algorithm* is covering for it.

**Comparing NaT (or NaN) with `==`.** `NaT == NaT` is False by IEEE discipline; membership
and filtering must use `np.isnat`/`np.isnan`. 02.5 builds the full missing-data toolkit.

**Fixed-width strings for open vocabularies.** `<U11` sized itself to today's longest status
and pads everything; the first longer value truncates on assignment into an existing array.
StringDType removes the ceiling and the padding — and `np.strings`, not `np.char`, is its
API.

**Auditing an upgrade only for exceptions.** The port's changes were one refusal
(OverflowError), one silent wrap, and one silent precision *gain*. A test suite that only
catches raises finds a third of the surface; an executed promotion table finds all of it.


## Interview Questions

1. **Derive this.** An int32 accumulator sums a column whose true total is T. Derive what
   the accumulator displays, and compute it for T = 1,552,569,768,898. *Answer shape:*
   two's-complement addition is arithmetic mod 2³²; the display is the representative of
   T mod 2³² in the signed window, here T − 361·2³² = 2,086,575,042 — plausible-looking,
   which is the danger.
2. **Design this.** Define the dtype policy for a billing pipeline: storage, aggregation,
   modelling, and the checks that enforce it. *Answer shape:* int64 minor units at rest;
   explicit int64 accumulators on reductions; float64 for modelling; contract at ingestion
   checking dtype, sum-headroom (1000×) and spacing-vs-resolution; promotion table in CI
   per pinned numpy.
3. **Debug this.** A dashboard number jumps 700-fold after a dependency upgrade; data
   hashes match and logs are clean in both eras. Walk your diagnosis. *Answer shape:* pin
   both environments and reproduce the pair on identical data; localise to the lane and the
   operation; compare the two results against an independently computed truth (wider
   accumulator, `math.fsum`, or archived reconciliation) to decide which era was wrong;
   name the mechanism before shipping either number.
4. State NEP 50's rule and give the three behaviour changes a 1.x→2.x port must audit.
   *Answer shape:* Python scalars are weak (adopt the array dtype), numpy scalars are
   strong (promote by type); values no longer vote. Audit: in-range unsigned arithmetic now
   wraps where 1.x widened; out-of-range Python scalars now raise; numpy float64 scalars now
   win promotions, adding precision.
5. Why did the same script give different revenue totals on Windows and Linux under numpy
   1.x, and what changed in 2.x? *Answer shape:* integer reductions accumulate in the
   default integer, the C long in 1.x — 32-bit on Windows, 64-bit on Linux — so only
   Windows wrapped; 2.x pinned the default to int64 everywhere. `np.long` still aliases the
   C long and resurrects the difference.
6. When is float32 a defensible dtype for financial data, and what must accompany it?
   *Answer shape:* as model-input storage where ~1e-7 relative noise is immaterial, never
   for ledgers or aggregation; accompany with widened accumulators (`dtype=np.float64` on
   sums), a spacing-vs-resolution check at the column's maximum, and a stated tolerance for
   any comparison.
7. Your test suite compares a vectorized sum against a streaming implementation and they
   agree on a thousand-row fixture but diverge by thousands at production scale. What
   happened? *Answer shape:* both are float32; numpy's pairwise tree keeps error ~O(log n)
   while the streaming loop accumulates ~O(n) error and starts absorbing small addends once
   the total dwarfs the spacing; the fixture was too small for the divergence to express.
   Fix the dtype (or the accumulator), not the tolerance.


## Key Takeaways

- Price dtypes in domain units before shipping a column: int32 is INR 21.5 million of
  paise, float32 near 1e8 spaces values 8 apart — both ceilings PayFlow crosses routinely.
- Integer overflow on aggregation is an *accumulator* property: elements that all fit can
  still lap the ceiling 361 times, silently, and the leftover looks like revenue.
- State accumulators explicitly on money paths (`sum(dtype=np.int64)`); the default just
  proved platform-dependent, and `np.long` is the C long, not a 64-bit guarantee.
- NEP 50 in one sentence — Python scalars weak, numpy scalars strong, values don't vote —
  and the port's audit surface is one silent wrap, one new refusal, one silent precision gain.
- A narrow dtype that "tests fine" may be shielded by its algorithm: pairwise summation hides
  float32's drift until a streaming rewrite removes the tree. Fix the dtype, not the tolerance.
- datetime64 is exact integer arithmetic with a unit in the type; NaT follows NaN discipline;
  pick the coarsest unit the domain needs.
- StringDType ends the fixed-width/object era — variable-length, heap-backed, with
  `np.strings` as its API — and resolves M6's casing in one vectorized call.
- Write the dtype contract where the data enters: dtype, sum-headroom, spacing — three cheap
  checks that fail a doomed column years before its month-end.


## Related

**Backward**

- **02.1 The ndarray Memory Model** — dtype as *interpretation* (bytes, views, `.view` vs
  `.astype`); this notebook is the same choice's *arithmetic* consequences.
- **01.2 First Contact with the PayFlow Data Universe** — the unit error (M2) is the sibling
  failure: units are meaning outside the machine, dtypes are meaning inside it, and both
  corrupt silently through an innocent `sum`.
- **01.3 Reproducibility as an Engineering Contract** — the manifest hashes that eliminated
  data in Diagnosis step 2, and the verify-by-running discipline the promotion table encodes.

**Forward**

- **02.5 Ufuncs, Reductions & Missing Data** — reductions in full: axis semantics, `out=`,
  NaN/NaT handling, and where accumulator dtypes surface again.
- **02.7 Memory Layout & Performance** — the legitimate case *for* narrow dtypes: memory
  bandwidth, cache, and when float32 features are the right trade.
- **03.1 Pandas & Modern DataFrames** — pandas 3's string dtype and nullable types build
  directly on this notebook's StringDType and NaT stories.
- **05 (math lane) numerical precision** — IEEE-754 derived properly: machine epsilon,
  cancellation, and why pairwise summation bounds error at O(log n).
